In [161]:
import gzip
import pandas as pd

# gene expression
gene_df = pd.read_csv(
    "../Data/GSE53482-GPL13667_series_matrix.txt.gz",
    sep="\t",
    comment='!'
)

# miRNA expression
mirna_df = pd.read_csv(
    "../Data/GSE53482-GPL14613_series_matrix.txt.gz",
    sep="\t",
    comment='!'
)

print(gene_df.shape)
print(mirna_df.shape)

(49386, 74)
(20212, 74)


In [162]:
gene_df = gene_df.set_index("ID_REF").T
mirna_df = mirna_df.set_index("ID_REF").T

print(gene_df.shape)
print(mirna_df.shape)

(73, 49386)
(73, 20212)


In [163]:
def extract_metadata(file_path):
    meta_rows = []
    sample_ids = None

    with gzip.open(file_path, 'rt') as f:
        for line in f:
            if line.startswith("!Sample_geo_accession"):
                sample_ids = line.strip().split("\t")[1:]
            
            if line.startswith("!Sample_characteristics_ch1"):
                values = line.strip().split("\t")[1:]
                meta_rows.append(values)

    meta = pd.DataFrame(meta_rows).T
    
    # assign correct sample IDs
    meta.index = sample_ids
    
    return meta

meta_gene = extract_metadata("../Data/GSE53482-GPL13667_series_matrix.txt.gz")
meta_mirna = extract_metadata("../Data/GSE53482-GPL14613_series_matrix.txt.gz")

meta_gene.head()

,0,1,2,3,4
"""GSM1294472""","""supplier: Vannucchi""","""cell type: MPD""","""disease: PMF""","""jak2v617f: neg""","""tissue: PB"""
"""GSM1294473""","""supplier: Vannucchi""","""cell type: MPD""","""disease: PMF""","""jak2v617f: pos""","""tissue: PB"""
"""GSM1294474""","""supplier: Vannucchi""","""cell type: MPD""","""disease: PMF""","""jak2v617f: pos""","""tissue: PB"""
"""GSM1294475""","""supplier: Vannucchi""","""cell type: MPD""","""disease: PMF""","""jak2v617f: pos""","""tissue: PB"""
"""GSM1294476""","""supplier: Vannucchi""","""cell type: MPD""","""disease: PMF""","""jak2v617f: pos""","""tissue: PB"""


In [164]:
def parse_metadata(meta_df):
    parsed_rows = []
    
    for idx, row in meta_df.iterrows():  # 👈 keep index
        row_dict = {}
        
        for item in row:
            if pd.isna(item):
                continue
            
            item = item.strip('"')
            
            key, value = item.split(": ")
            row_dict[key] = value
        
        parsed_rows.append(row_dict)
    
    parsed_df = pd.DataFrame(parsed_rows)
    
    parsed_df.index = meta_df.index
    
    parsed_df.index = parsed_df.index.str.strip('"')

    return parsed_df

meta_gene_clean = parse_metadata(meta_gene)
meta_gene_clean.head()

,supplier,cell type,disease,jak2v617f,tissue
GSM1294472,Vannucchi,MPD,PMF,neg,PB
GSM1294473,Vannucchi,MPD,PMF,pos,PB
GSM1294474,Vannucchi,MPD,PMF,pos,PB
GSM1294475,Vannucchi,MPD,PMF,pos,PB
GSM1294476,Vannucchi,MPD,PMF,pos,PB


In [165]:
meta_gene_clean.columns = [
    col.strip().replace(" ", "_").lower()
    for col in meta_gene_clean.columns
]

meta_gene_clean.head()


,supplier,cell_type,disease,jak2v617f,tissue
GSM1294472,Vannucchi,MPD,PMF,neg,PB
GSM1294473,Vannucchi,MPD,PMF,pos,PB
GSM1294474,Vannucchi,MPD,PMF,pos,PB
GSM1294475,Vannucchi,MPD,PMF,pos,PB
GSM1294476,Vannucchi,MPD,PMF,pos,PB


In [166]:
meta_gene_clean["label"] = meta_gene_clean["disease"].apply(
    lambda x: 1 if x == "PMF" else 0
)

meta_gene_clean.head()
meta_gene_clean["label"].value_counts()

label
1    42
0    31
Name: count, dtype: int64

In [167]:
print(gene_df.shape)
print(meta_gene_clean.shape)

(73, 49386)
(73, 6)


In [168]:
assert gene_df.shape[0] == meta_gene_clean.shape[0]

In [ ]:
X = gene_df
y = meta_gene_clean["label"].values

print(X.shape)
print(len(y))

(73, 49386)
73


In [170]:
# Check alignment
assert gene_df.index.equals(meta_gene_clean.index)

# Check class balance
import numpy as np
print(np.bincount(y))

# Check duplicates
print("Duplicate samples:", gene_df.T.duplicated().sum())

# Check zero-variance features
print("Zero variance features:", np.sum(np.std(X, axis=0) == 0))

[31 42]
Duplicate samples: 0
Zero variance features: 0


In [171]:
print(gene_df.index[:10])
print(meta_gene_clean.index[:10])

Index(['GSM1294472', 'GSM1294473', 'GSM1294474', 'GSM1294475', 'GSM1294476',
       'GSM1294477', 'GSM1294478', 'GSM1294479', 'GSM1294480', 'GSM1294481'],
      dtype='object')
Index(['GSM1294472', 'GSM1294473', 'GSM1294474', 'GSM1294475', 'GSM1294476',
       'GSM1294477', 'GSM1294478', 'GSM1294479', 'GSM1294480', 'GSM1294481'],
      dtype='object')


In [172]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("feature_selection", SelectKBest(score_func=f_classif, k=500)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

In [181]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc")

print("Regression ROC-AUC mean:", scores.mean())
print("Regression ROC-AUC std:", scores.std())

y_random = np.random.permutation(y)

shuffle_scores = cross_val_score(pipeline, X, y_random, cv=cv, scoring="roc_auc")

print("Regression Shuffled ROC-AUC:", shuffle_scores.mean())
print("Regression Shuffled ROC-AUC std:", shuffle_scores.std())

Regression ROC-AUC mean: 1.0
Regression ROC-AUC std: 0.0
Regression Shuffled ROC-AUC: 0.5353174603174603
Regression Shuffled ROC-AUC std: 0.18674494303550798


In [180]:
print(gene_df.index[:5])
print(meta_gene_clean.index[:5])

Index(['GSM1294472', 'GSM1294473', 'GSM1294474', 'GSM1294475', 'GSM1294476'], dtype='object')
Index(['GSM1294472', 'GSM1294473', 'GSM1294474', 'GSM1294475', 'GSM1294476'], dtype='object')


In [176]:
feature_names = gene_df.index 

In [177]:
from sklearn.ensemble import RandomForestClassifier

# Define Random Forest
rf = RandomForestClassifier(random_state=42)

# Define 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate with cross-validation
scores = cross_val_score(rf, X, y, cv=cv, scoring="roc_auc")

print("Random Forest ROC-AUC mean:", scores.mean())
print("Random Forest ROC-AUC std:", scores.std())

# Optional: Shuffle test to ensure model isn't learning noise
y_random = np.random.permutation(y)
shuffle_scores = cross_val_score(rf, X, y_random, cv=cv, scoring="roc_auc")

print("Random Forest (shuffled labels) ROC-AUC mean:", shuffle_scores.mean())
print("Random Forest (shuffled labels) ROC-AUC std:", shuffle_scores.std())

Random Forest ROC-AUC mean: 1.0
Random Forest ROC-AUC std: 0.0
Random Forest (shuffled labels) ROC-AUC mean: 0.49074074074074076
Random Forest (shuffled labels) ROC-AUC std: 0.13817718888485434


In [178]:
importances = model.feature_importances_

importances = model.feature_importances_  # length = X.shape[1]

feat_imp = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feat_imp.head(10)

,feature,importance
24387,11739487_s_at,0.018482
30862,11745962_a_at,0.017040
42524,11757624_s_at,0.015590
44546,11759646_at,0.014654
44653,11759753_a_at,0.010558
40512,11755612_s_at,0.010000
23925,11739025_a_at,0.010000
41234,11756334_x_at,0.010000
15632,11730732_at,0.010000
3677,11718777_at,0.010000
